## Installation

In [1]:
#!brew install poppler

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

## Test Extractors

### Rubric Extractor

In [33]:
import fitz  # PyMuPDF

def extract_rubric(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return text.strip()

In [34]:
rubric = extract_rubric("../data/ExamQuestionAndRubric.pdf")

In [35]:
rubric

'Exam question \nCase study (3 points): In this exercise, your task is to apply the Bad Actors Modeling strategy to the \nSmartCity solution described in the scenario below: \n• \nChoose 3 different categories among the five in the Bad Actors Modeling strategy (money, \npolitics, entertainment, ideas, self-interest).  \n• \nThen for each of the categories you have chosen: identify and describe one harmful action a \nbad actor could take and how it could negatively impact stakeholders (1-2 sentences for \neach). Actions must be different from each other.  \nScenario: The Swiss company SmartCity wants to help local authorities to better manage city \ninfrastructures and resources by deploying a solution based on the Internet of Things (IoT) - a \nnetwork of objects with computing capabilities connected to the Internet through Wi-Fi or 5G. The \nsolution proposed by SmartCity uses a range of connected devices to monitor waste-bins, traffic flow \nas well as weather conditions throughout a

'Exam question \nCase study (3 points): In this exercise, your task is to apply the Bad Actors Modeling strategy to the \nSmartCity solution described in the scenario below: \n• \nChoose 3 different categories among the five in the Bad Actors Modeling strategy (money, \npolitics, entertainment, ideas, self-interest).  \n• \nThen for each of the categories you have chosen: identify and describe one harmful action a \nbad actor could take and how it could negatively impact stakeholders (1-2 sentences for \neach). Actions must be different from each other.  \nScenario: The Swiss company SmartCity wants to help local authorities to better manage city \ninfrastructures and resources by deploying a solution based on the Internet of Things (IoT) - a \nnetwork of objects with computing capabilities connected to the Internet through Wi-Fi or 5G. The \nsolution proposed by SmartCity uses a range of connected devices to monitor waste-bins, traffic flow \nas well as weather conditions throughout a city. Level monitoring sensors on waste-bins allow waste \ncollection services to optimize their routes. Cameras on major roads and intersections are connected \nto machine learning models to analyze real-time traffic data and dynamically adjust green light times. \nUsing real-time weather data, indoor ventilation and temperature are automatically adjusted to keep \nbuilding occupants comfortable while minimizing energy use. Weather data is also used in predictive \nanalytics to identify health risks and aggregated information is communicated to local communities in \na public app, for instance in case of air pollution or heatwave. Staff from the city authorities can \nmonitor and control the system remotely from a centralized dashboard. SmartCity plans to roll out its \nsolution in a number of European countries with large cities, including France, Germany, Italy and \nSpain.  \n \nGrading rubric \n1 point per bad actor action (total = 3 points) \n• \n0.5 the action described corresponds to the category (check the cheatsheet)  \n• \n0.5 the description makes clear how the action is harmful / negatively impacts stakeholders \nand is specific to SmartCity (i.e. adapted to the scenario)   \nPenalizations:  \n• \n-0.5 if two actions are too similar to each other  \n• \n-0.25 if negative impact is not sufficiently described'

### Student Answer Extractor

In [13]:
import pytesseract
from pdf2image import convert_from_path

def extract_student_answers(paths):
    results = []

    for path in paths:
        pages = convert_from_path(path)
        text = ""

        for page in pages:
            text += pytesseract.image_to_string(page)

        results.append(text.strip())

    return results

In [30]:

from openai import OpenAI
import fitz
import base64
import json
import os

client = OpenAI()


def pdf_to_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    images = []

    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img_bytes = pix.tobytes("png")
        b64 = base64.b64encode(img_bytes).decode("utf-8")
        images.append(f"data:image/png;base64,{b64}")

    return images
def extract_student_transcription(pdf_paths, output_file="student_transcriptions.json"):
    results = []

    for path in pdf_paths:
        images = pdf_to_images(path)

        response = client.responses.create(
            model="gpt-4.1-mini",
            input=[{
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """
You are a transcription system for handwritten exam answers.

IMPORTANT:
- Do NOT summarize
- Do NOT interpret
- Do NOT improve grammar
- Preserve structure
- Use [unclear] if needed

Return STRICT JSON:

{
  "transcription": "...",
  "uncertain_words": [],
  "confidence": 0-100
}
"""
                    },
                    *[
                        {
                            "type": "input_image",
                            "image_url": img,
                            "detail": "high"
                        }
                        for img in images
                    ]
                ]
            }]
        )

        raw_output = response.output_text

        try:
            parsed = json.loads(raw_output)
        except:
            parsed = {
                "transcription": raw_output,
                "uncertain_words": [],
                "confidence": 0
            }

        results.append({
            "file": path,
            "result": parsed
        })

    # 💾 SAVE TO FILE
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    return results

In [31]:
students = extract_student_transcription([
        "../data/StudentAnswer-Student1.pdf",
        "../data/StudentAnswer-Student2.pdf",
        "../data/StudentAnswer-Student3.pdf"
    ])

In [28]:
students

{'transcription': 'Money: A competitor of the company Smartcity can hack the centralized monitoring system, this would lead to a mess with the traffic and the bins. The company Smartcity would be negatively impacted in their reputation and this would lead to the loss of money by not signing contract with other European countries. The motivation for the competitor is the money they gain some part of the market.\nPolitics: A competitor of the current "maire (chef de la ville)" can disturb the sensors on waste-bins, the harm would be that citizens wouldn\'t be happy anymore with the council of the city because there would be bins everywhere due to the disruption of the sensors. The motivation for the competitor would be to be elected in the next elections.\nIdeas: A group of people pro-privacy can destroy/cover the cameras used to analyze the real-time traffic arguing that it is not in their ideology to be looked after by ML augmented cameras. The stakeholders harmed would be all the user

#### OCR Extraction

In [24]:
students

['Politics Tictillig amen Cam Vee\n| [ws pero fe ihe ud ji bet\nom y . bt a a7\n\nbeg a have ey pad Pecan\nHe Robhas drm tube Hue vwheable\n\nWeld weticle - furor’, Lomeas , Hutter\nwae be soll Huon oe Oh a “a\ntrtibaiebuat Y Some bad deler aol\nHe wore ellecton ewer bY many ula\nde. Pauses for mahe Hw Wwarte~colle ae,\nLrteot Hu lallredors be He Come Lacobialen\n\nSeren [yes',
 '© Cateye “F Z vl ene?\n\naH lef 7 | waa te bo lA i a or even 14 ulingy iC\n\n6S iMce pl mig at 4 Bowe esol l eS ets Se vf\n\nOr it cen eucoy Tehe dre on authori fies te speek\noN lod “| ia me a\n\n‘ Catyer7 . Sel “interest | | .\n\nSome pepe might vse ot for dl puaformedon | sme\nnt can pobtly veh peed, . They could\niw he posped ea the wes cud ollow {he ao les |\nput could hetm Lee fj” Qoeng le 1 t fetsen\n\nAs tet eM\nwhe might hewve Ce oli kt Ly 2 bave Toe street\n\nent eu yt anil Freve le fi" yre Th We\nond Jem rere Wee being adj Ktedh mutters ie My wih\nagen dence on Thee weefler outside o¢ The enviunat\n

### Teaching Material Extractor

In [6]:
import fitz
import base64
import json
import os
from datetime import datetime
from openai import OpenAI

client = OpenAI()


# ----------------------------
# 1. Convert PDF → images
# ----------------------------
def pdf_to_base64_images(pdf_path: str):
    doc = fitz.open(pdf_path)
    images = []

    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        base64_img = base64.b64encode(img_bytes).decode("utf-8")

        images.append({
            "type": "input_image",
            "image_url": f"data:image/png;base64,{base64_img}",
            "detail": "high"
        })

    return images


# ----------------------------
# 2. Extract structured content
# ----------------------------
def extract_teaching_material(pdf_path: str, save_output: bool = True):

    images = pdf_to_base64_images(pdf_path)

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": """
You are extracting teaching material from a PDF.

IMPORTANT:
- Do NOT summarize.
- Extract EVERYTHING visible.
- Preserve structure exactly.
- Include diagram content (WENN diagrams, arrows, boxes).
- If text is in a figure, transcribe it.
- If relationships exist, describe them explicitly.

OUTPUT FORMAT (strict JSON):

{
  "key_concepts": [],
  "definitions": [],
  "diagram_explanations": [],
  "relationships": [],
  "raw_notes": []
}

Rules:
- Be exhaustive
- Do not omit small details
- Do not interpret beyond visible information
"""
                },
                *images
            ]
        }]
    )

    extracted_text = response.output_text

    # ----------------------------
    # 3. Save output for inspection
    # ----------------------------
    if save_output:
        os.makedirs("outputs", exist_ok=True)

        filename = f"outputs/teaching_material_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        try:
            parsed = json.loads(extracted_text)
        except Exception:
            parsed = {"raw_output": extracted_text}

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(parsed, f, indent=2, ensure_ascii=False)

        print(f"[Saved] {filename}")

    return extracted_text

In [4]:
teaching = extract_teaching_material("../data/TeachingResource-BAD_ACTORS_STRATEGY.pdf")

In [5]:
teaching

'- **Fundamentals of the strategy**\n\n  - **Goal**:  \n    - Minimize vulnerabilities from the beginning of software design through risk assessment and mitigation.\n\n  - **When to use it**:  \n    - During the software design phase  \n    - Prior to any updates  \n    - During audits and inspections  \n    - As part of incident response  \n    - While improving the software\n\n- **What is a Bad actor?**\n\n  - Definition:  \n    - A "bad actor" or "threat actor" is a person or group that tries to exploit a system to achieve their own personal goals.\n\n  - Key points:  \n    - Multiple types of bad actors exist.  \n    - They are driven by different motivations.  \n    - Understanding and categorizing these motivations is essential for threat modeling.\n\n- **Assessment questions** (to guide risk analysis)\n\n  - Who would want to misuse the software or use it in an unintended way?  \n  - What would be the consequences of such misuse?  \n  - How could the negative consequences be pre

## Test pipeline

In [47]:
def call_llm_text(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4, #diversity,
        response_format={"type": "json_object"}
    )
    return response.choices[0].message.content

#### Analyze Rubric

In [ ]:
def analyze_rubric(rubric, teaching_material):
    prompt = f"""
You are an expert in exam design.

Your task is to evaluate how well the rubric aligns with the teaching material.

RUBRIC:
{rubric}

TEACHING MATERIAL:
{teaching_material}

IMPORTANT:
- Focus on alignment with teaching
- Identify missing concepts
- Identify unclear grading rules
- Do NOT invent new criteria

Evaluate it on:

1. Ambiguity → Is the rubric formulated with the objective, unambiguous criteria?
2. Applicability → Does the rubric cover the diversity of possible student responses?
3. Discrimination → Does the rubric clearly separate excellent from poor work?

Return STRICT JSON:

{{
  "ambiguity_issues": ["..."],
  "applicability_issues": ["..."],
  "discrimination_issues": ["..."],
  "alignment_issues": ["..."]
}}
"""
    output = call_llm_text(prompt)
    return extract_json(output)

In [ ]:
rubric_analysis = analyze_rubric(rubric, teaching)

In [ ]:
rubric_analysis

{'ambiguity_issues': ["The rubric refers to 'check the cheatsheet' for matching actions to categories, but the exam does not provide a cheatsheet or specify if the teaching material serves as one.",
  "The criterion '-0.5 if two actions are too similar to each other' is vague; it does not define what constitutes 'too similar', leaving room for subjective interpretation.",
  "The requirement for the negative impact to be 'sufficiently described' is not operationalized—no guidance on what is 'sufficient'.",
  'It is not specified whether partial credit can be given if an action is only partially adapted to the scenario.'],
 'applicability_issues': ['The rubric expects one action per category, but the teaching material notes that motivation categories overlap and that some actions may fit multiple categories, which may cause confusion for students who justify their choices differently.',
  'The rubric does not address how to grade if a student provides more than three categories or action

### Simulate Grading

In [126]:
import json
import re
#from app.llm import call_llm_text


def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group())


def grade_student_answer(rubric, answer):
    prompt = f"""
You are a strict and consistent grader.

IMPORTANT:
- Output MUST be valid JSON
- You MUST use ONLY the rubric below
- Do NOT introduce external knowledge
- If the rubric is unclear, explicitly say so

RUBRIC:
{rubric}

STUDENT ANSWER:
{answer}

TASK:
1. Extract ALL criteria from the rubric
2. Evaluate each criterion separately
3. Assign score strictly within defined max score
4. If a criterion is unclear → mark it as "ambiguous" -> is_ambiguous = true


Return STRICT JSON:

{{
  "criteria_scores": [
    {{
      "criterion": "string",
      "max_score": number,
      "score": number,
      "reason": "string",
      "is_ambiguous": true
    }}
  ],
  "final_grade": number,
  "overall_reason": "string"
}}
"""

    output = call_llm_text(prompt)
    try:
        return extract_json(output)
    except Exception as e:
        return {
            "criteria_scores": [],
            "final_grade": 0,
            "overall_reason": "Parsing failed",
            "error": str(e),
            "raw_output": output
        }


def simulate_grading(rubric, students, runs=3):
    """
    Run grading multiple times to measure consistency (ambiguity).
    """
    all_runs = []

    for _ in range(runs):
        run_results = []
        for s in students:
            grade = grade_student_answer(rubric, s)
            run_results.append(grade)
        all_runs.append(run_results)

    return all_runs

In [127]:
grading_runs = simulate_grading(rubric, students)

In [128]:
grading_runs

[[{'criteria_scores': [{'criterion': 'Politics action corresponds to category (0.5)',
     'max_score': 0.5,
     'score': 0.5,
     'reason': "The action describes intelligence agents using the system for espionage, which fits the 'politics' category.",
     'is_ambiguous': False},
    {'criterion': 'Politics action negative impact is specific and clear (0.5)',
     'max_score': 0.5,
     'score': 0.5,
     'reason': 'The answer specifies loss of privacy and security for countries and their populations due to surveillance via SmartCity, which is a clear and scenario-specific negative impact.',
     'is_ambiguous': False},
    {'criterion': 'Money action corresponds to category (0.5)',
     'max_score': 0.5,
     'score': 0.5,
     'reason': "The action involves robbers stealing valuable equipment for resale, which is motivated by financial gain and fits the 'money' category.",
     'is_ambiguous': False},
    {'criterion': 'Money action negative impact is specific and clear (0.5)',
  

### Generate Synthetic Answers

In [129]:
#from app.llm import call_llm_text
import json
import re


def extract_json(text):
    return json.loads(re.search(r"\{.*\}", text, re.DOTALL).group())


def generate_synthetic_answers(teaching, rubric):
    """
    Generates calibration dataset:
    - good answers
    - bad answers
    - edge cases
    """

    prompt = f"""
You are generating synthetic student answers for rubric evaluation.

TEACHING MATERIAL:
{teaching}

RUBRIC:
{rubric}

Generate:

1. 4 HIGH-QUALITY answers (full understanding)
2. 4 LOW-QUALITY answers (misunderstanding / missing parts)
3. 3 EDGE CASE answers (partially correct but ambiguous)

Return STRICT JSON:

{{
  "good": ["..."],
  "bad": ["..."],
  "edge": ["..."]
}}
"""

    return extract_json(call_llm_text(prompt))

In [130]:
synthetic = generate_synthetic_answers(teaching, rubric)


In [131]:
synthetic

{'good': ["1. Money: A criminal group hacks into the waste-bin sensors and manipulates the data to make it appear that bins are always full, causing the city to pay for unnecessary waste collection services, resulting in financial loss for local authorities.\n2. Politics: A nation-state actor tampers with the traffic camera data to create artificial congestion during an election day, disrupting voter turnout and undermining trust in the city's ability to manage infrastructure.\n3. Self-Interest: A city employee with dashboard access changes indoor climate settings in public buildings to make them uncomfortable for a rival department, negatively affecting staff productivity and well-being.",
  "1. Ideas: Hacktivists deface the public app with misleading air quality alerts to promote their environmental agenda, causing public panic and eroding trust in city communications.\n2. Entertainment: Thrill-seekers remotely trigger false weather alerts through the IoT system, leading to unnecessa

In [132]:
synthetic_good = synthetic["good"]
synthetic_bad = synthetic["bad"]

In [144]:
synthetic_edge = synthetic["edge"]

In [133]:

good_runs = simulate_grading(rubric, synthetic_good)
bad_runs = simulate_grading(rubric, synthetic_bad)

good_scores = [g["final_grade"] for run in good_runs for g in run]
bad_scores = [g["final_grade"] for run in bad_runs for g in run]

In [145]:
edge_runs = simulate_grading(rubric,synthetic_edge)
edge_scores = [g["final_grade"] for run in edge_runs for g in run]


In [134]:
good_runs

[[{'criteria_scores': [{'criterion': 'Money action corresponds to category',
     'max_score': 0.5,
     'score': 0.5,
     'reason': "The action (criminal group manipulates waste-bin sensor data for financial gain) fits the 'money' category.",
     'is_ambiguous': False},
    {'criterion': 'Money action harmful impact is specific to SmartCity',
     'max_score': 0.5,
     'score': 0.5,
     'reason': 'The negative impact (financial loss for local authorities due to unnecessary waste collection) is clearly described and specific to the SmartCity scenario.',
     'is_ambiguous': False},
    {'criterion': 'Politics action corresponds to category',
     'max_score': 0.5,
     'score': 0.5,
     'reason': "The action (nation-state actor tampers with traffic data to disrupt election) fits the 'politics' category.",
     'is_ambiguous': False},
    {'criterion': 'Politics action harmful impact is specific to SmartCity',
     'max_score': 0.5,
     'score': 0.5,
     'reason': 'The negative i

In [135]:
good_scores

[3, 3, 3, 3, 3, 3, 3, 2.75, 3, 3, 3, 3.0]

In [137]:
bad_runs

[[{'criteria_scores': [{'criterion': 'Action corresponds to the chosen category (0.5 per action, max 1.5)',
     'max_score': 1.5,
     'score': 1.0,
     'reason': "Money: The action (hacking to steal money) fits the 'money' category. Politics: Changing traffic lights for political reasons fits 'politics.' Entertainment: Using cameras for fun is only loosely connected to 'entertainment,' as the action is not clearly described as being for entertainment purposes (e.g., pranks, livestreaming, etc.), so partial credit.",
     'is_ambiguous': False},
    {'criterion': 'Description makes clear how the action is harmful/negatively impacts stakeholders and is specific to SmartCity (0.5 per action, max 1.5)',
     'max_score': 1.5,
     'score': 0.75,
     'reason': "Money: The description lacks detail on how stealing money via hacking the SmartCity system would harm stakeholders (e.g., city funds lost, services disrupted), so partial credit. Politics: The negative impact is not described (e.

In [136]:
bad_scores

[1.25, 1.25, 1.25, 2.25, 1.5, 1.25, 1.25, 2.0, 0.75, 0.75, 1.5, 1.0]

In [146]:
edge_runs

[[{'criteria_scores': [{'criterion': 'Action corresponds to the chosen category (0.5 per action)',
     'max_score': 1.5,
     'score': 1.5,
     'reason': "All three actions (money, politics, entertainment) are plausibly mapped to their respective categories according to the rubric's cheatsheet.",
     'is_ambiguous': False},
    {'criterion': 'Description makes clear how the action is harmful/negatively impacts stakeholders and is specific to SmartCity (0.5 per action)',
     'max_score': 1.5,
     'score': 0.75,
     'reason': "For 'Money', the negative impact on stakeholders is not clearly described. For 'Politics', the impact on SmartCity is vague. For 'Entertainment', the negative impact is not fully explained. Each of these is penalized -0.25, as per the rubric.",
     'is_ambiguous': False},
    {'criterion': 'Actions must be different from each other (penalize -0.5 if two are too similar)',
     'max_score': 0,
     'score': 0,
     'reason': 'The actions are different from ea

In [147]:
edge_scores

[2.25, 1.75, 0.75, 2.25, 1.5, 1.5, 2.25, 2.25, 0.75]

## Test Evaluation

### Metrics

In [148]:
import numpy as np


# =========================================================
# 1. AMBIGUITY → criterion-level disagreement
# =========================================================

def ambiguity_score(grades_runs):
    """
    Measures disagreement across graders at criterion level.
    """

    if len(grades_runs) < 2:
        return 100

    base_run = grades_runs[0]
    total_diff = 0
    total = 0

    for run in grades_runs[1:]:
        for g1, g2 in zip(base_run, run):

            c1 = {c["criterion"]: c["score"] for c in g1["criteria_scores"]}
            c2 = {c["criterion"]: c["score"] for c in g2["criteria_scores"]}

            keys = set(c1.keys()) | set(c2.keys())

            for k in keys:
                total += 1
                total_diff += abs(c1.get(k, 0) - c2.get(k, 0))

    if total == 0:
        return 100

    avg_diff = total_diff / total

    # convert to 0-100
    return max(0, 100 - avg_diff * 100)



# =========================================================
# 2. DISCRIMINATION → ranking + separation
# =========================================================

def discrimination_score(good_scores, edge_scores, bad_scores):
    """
    Measures whether rubric preserves correct ranking:
    good > edge > bad
    """

    def mean(x):
        return sum(x) / (len(x) + 1e-8)

    g = mean(good_scores)
    e = mean(edge_scores)
    b = mean(bad_scores)

    # ranking correctness
    correct_order = int(g > e > b)

    # margins (important!)
    margin_ge = g - e
    margin_eb = e - b

    margin_score = max(0, (margin_ge + margin_eb) * 50)

    return min(100, correct_order * 50 + margin_score)



# =========================================================
# 3. APPLICABILITY → semantic coverage + edge handling
# =========================================================

def applicability_score(grades_runs, edge_scores):
    """
    Measures whether rubric meaningfully handles:
    - good cases
    - bad cases
    - edge cases
    """

    used_criteria = set()

    for run in grades_runs:
        for g in run:
            for c in g["criteria_scores"]:
                used_criteria.add(c["criterion"])

    # criterion richness
    criterion_score = min(100, len(used_criteria) * 8)

    # edge case sensitivity
    edge_mean = sum(edge_scores) / (len(edge_scores) + 1e-8)

    edge_score = min(100, edge_mean * 20)

    # combine both signals
    return 0.6 * criterion_score + 0.4 * edge_score



# =========================================================
# OPTIONAL: FINAL SCORE
# =========================================================

def final_score(a, b, c):
    return round((a + b + c) / 3, 2)

In [ ]:


def detect_rubric_failures(grading_runs, good_scores, edge_scores, bad_scores):

    ambiguity = ambiguity_score(grading_runs)
    discrimination = discrimination_score(good_scores, edge_scores, bad_scores)
    applicability = applicability_score(grading_runs, edge_scores)

    failures = []

    # -------------------------
    # AMBIGUITY
    # -------------------------
    if ambiguity < 80:
        failures.append({
            "type": "ambiguity",
            "message": "Grading is inconsistent across runs.",
            "score": ambiguity
        })

    # -------------------------
    # DISCRIMINATION
    # -------------------------
    if discrimination < 70:
        failures.append({
            "type": "discrimination",
            "message": "Rubric fails to reliably rank good > edge > bad.",
            "score": discrimination
        })

    # extra diagnostic: inverted ranking
    if good_scores and bad_scores and sum(good_scores)/len(good_scores) <= sum(bad_scores)/len(bad_scores):
        failures.append({
            "type": "critical_discrimination_failure",
            "message": "Bad answers score equal or higher than good answers.",
            "score": discrimination
        })

    # -------------------------
    # APPLICABILITY
    # -------------------------
    if applicability < 70:
        failures.append({
            "type": "applicability",
            "message": "Rubric does not handle full diversity of answer types.",
            "score": applicability
        })

    return {
        "scores": {
            "ambiguity": round(ambiguity, 2),
            "discrimination": round(discrimination, 2),
            "applicability": round(applicability, 2)
        },
        "failures": failures
    }

In [151]:
evaluation = detect_rubric_failures(grading_runs, good_scores, edge_scores, bad_scores)

In [152]:
evaluation

{'scores': {'ambiguity': 65.12, 'discrimination': 100, 'applicability': 73.56},
 'failures': [{'type': 'ambiguity',
   'message': 'Grading is inconsistent across runs.',
   'score': 65.11627906976744}]}

## Build Prompt for Rubric Improvement

In [153]:
def build_prompt(rubric, teaching, students, grading_runs, evaluation, rubric_analysis):

    student_block = "\n\n".join(
        [f"Student {i+1}:\n{s}" for i, s in enumerate(students)]
    )

    return f"""
You are an expert exam designer.

TASK:
Improve the grading rubric using MULTIPLE sources of evidence baesd on the following goals.

--- INPUTS ---

RUBRIC:
{rubric}

RUBRIC ANALYSIS:
{rubric_analysis}

TEACHING MATERIAL:
{teaching}

STUDENT ANSWERS:
{student_block}

GRADING RESULTS (multiple runs):
{grading_runs}

EVALUATION:
{evaluation}

FAILURES DETECTED:
{evaluation["failures"]}


Improve the rubric by:
- Reducing ambiguity (make criteria precise)
- Increasing applicability (cover more answer types)
- Improving discrimination (differentiate quality levels)

Goals:
1. Ambiguity → All the graders reach the same interpretation independently.
2. Applicability → No valid answer type is left unaddressed by the rubric.
3. Discrimination → High-quality answers score significantly better than weak ones.

OUTPUT STRICT JSON:
{{
  "improved_rubric": "...",
  "explanation": {{
    "ambiguity": "...",
    "applicability": "...",
    "discrimination": "..."
  }}
}}
"""

In [154]:
prompt = build_prompt(
        rubric, 
        teaching, 
        students, 
        grading_runs, 
        evaluation,
        rubric_analysis
        )

In [155]:
output = call_llm_text(prompt)

In [157]:
improved = extract_json(output)

In [158]:
improved

{'improved_rubric': "Grading Rubric for Bad Actors Modeling (Total: 3 points)\n\nGeneral Requirements:\n- The student must select three distinct motivation categories from the five defined in the Bad Actors Modeling strategy: Money, Politics, Entertainment, Self-Interest, Ideas. (Partial or full overlap is acceptable if justified; see below.)\n- For each category, the student must: (a) identify a plausible bad actor action motivated by that category, (b) describe how the action could negatively impact stakeholders, and (c) ensure the action is adapted to the SmartCity scenario.\n\nScoring per Category (1 point each, max 3):\nEach category is scored out of 1 point, subdivided as follows:\n\n1. Action-Category Match (0.3 points):\n   - 0.3: The action is clearly motivated by the selected category, as defined in the teaching material. If the action plausibly fits more than one category, the student must justify their choice (e.g., 'This is primarily about money because...').\n   - 0.15: T

In [159]:
print(improved["improved_rubric"])

Grading Rubric for Bad Actors Modeling (Total: 3 points)

General Requirements:
- The student must select three distinct motivation categories from the five defined in the Bad Actors Modeling strategy: Money, Politics, Entertainment, Self-Interest, Ideas. (Partial or full overlap is acceptable if justified; see below.)
- For each category, the student must: (a) identify a plausible bad actor action motivated by that category, (b) describe how the action could negatively impact stakeholders, and (c) ensure the action is adapted to the SmartCity scenario.

Scoring per Category (1 point each, max 3):
Each category is scored out of 1 point, subdivided as follows:

1. Action-Category Match (0.3 points):
   - 0.3: The action is clearly motivated by the selected category, as defined in the teaching material. If the action plausibly fits more than one category, the student must justify their choice (e.g., 'This is primarily about money because...').
   - 0.15: The action is weakly related, or 

In [83]:
improved["explanation"]

{'ambiguity': "The improved rubric reduces ambiguity by: (1) Defining each scoring sub-criterion with precise language and point values (0, 0.10, 0.25), (2) Providing explicit definitions for what constitutes a 'clear fit', 'partial fit', or 'no fit' for each criterion, (3) Stating how to handle overlap, extra categories, and non-standard examples, (4) Including a summary table for quick reference, and (5) Requiring justification if an action fits multiple categories.",
 'applicability': 'Applicability is increased by: (1) Allowing for justified overlap between categories, (2) Accepting logically consistent but non-standard examples if justified, (3) Specifying how to handle answers with more than three actions/categories, (4) Providing partial credit for partially correct or incomplete answers, and (5) Including both system security and user safety in the depth/insight criterion.',
 'discrimination': "Discrimination is improved by: (1) Using four sub-criteria per action, each with thr

## Test Human Review

In [160]:
import json

def human_review(improved_result):
    """
    Human can edit final rubric + explanation.
    """

    print("\n--- IMPROVED RUBRIC ---\n")
    print(improved_result["improved_rubric"])

    print("\n--- EXPLANATION ---\n")
    print(json.dumps(improved_result["explanation"], indent=2))

    print("\nDo you want to modify? (y/n)")
    choice = input("> ")

    if choice.lower() != "y":
        return improved_result

    new_rubric = input("\nNew rubric:\n")
    new_expl = input("\nNew explanation JSON:\n")

    try:
        new_expl = json.loads(new_expl)
    except:
        new_expl = improved_result["explanation"]

    return {
        "improved_rubric": new_rubric,
        "explanation": new_expl,
        "human_verified": True
    }

In [ ]:
human_review(improved)

In [ ]:
human_reviewed_rubric = human_review(improved)


--- IMPROVED RUBRIC ---

Grading Rubric for Bad Actors Modeling (Total: 3 points)

General Requirements:
- The student must select three distinct motivation categories from the five defined in the Bad Actors Modeling strategy: Money, Politics, Entertainment, Self-Interest, Ideas. (Partial or full overlap is acceptable if justified; see below.)
- For each category, the student must: (a) identify a plausible bad actor action motivated by that category, (b) describe how the action could negatively impact stakeholders, and (c) ensure the action is adapted to the SmartCity scenario.

Scoring per Category (1 point each, max 3):
Each category is scored out of 1 point, subdivided as follows:

1. Action-Category Match (0.3 points):
   - 0.3: The action is clearly motivated by the selected category, as defined in the teaching material. If the action plausibly fits more than one category, the student must justify their choice (e.g., 'This is primarily about money because...').
   - 0.15: The act

## Test Improved Rubric

#### Run both rubrics

In [86]:
improved_rubric = improved["improved_rubric"]

In [88]:
print(improved_rubric)

Grading Rubric for Bad Actors Modeling (Total: 3 points)

Overview:
Each student must select 3 different categories from the Bad Actors Modeling strategy (money, politics, entertainment, ideas, self-interest). For each, they must:
- Identify a plausible harmful action by a bad actor motivated by that category, adapted to the SmartCity scenario.
- Clearly describe the negative impact on specific stakeholders.

Scoring (per action/category, up to 1 point each):

For each of the 3 actions (repeat for each):
1. Category Alignment (0.25 points)
   - 0.25: The action is clearly motivated by the selected category, as defined in the teaching material (see definitions/examples). Overlap is acceptable if the student justifies their reasoning.
   - 0.10: The action is only loosely connected to the category, or the motivation is ambiguous, but a plausible justification is provided.
   - 0: The action does not fit the selected category and no reasonable justification is given.

2. Action Specificit

In [84]:
baseline_runs = grading_runs

In [85]:
baseline_runs

[[{'criteria_scores': [{'criterion': 'Politics action corresponds to category (0.5)',
     'max_score': 0.5,
     'score': 0.5,
     'reason': 'The action (intelligence agents using SmartCity to gain information about other countries) fits the politics category.',
     'is_ambiguous': False},
    {'criterion': 'Politics action negative impact is clear and scenario-specific (0.5)',
     'max_score': 0.5,
     'score': 0.5,
     'reason': 'The answer specifies loss of privacy and security for countries and their populations, which is a clear and scenario-specific negative impact.',
     'is_ambiguous': False},
    {'criterion': 'Money action corresponds to category (0.5)',
     'max_score': 0.5,
     'score': 0.5,
     'reason': 'The action (robbers stealing valuable installed materials for resale or use) fits the money category.',
     'is_ambiguous': False},
    {'criterion': 'Money action negative impact is clear and scenario-specific (0.5)',
     'max_score': 0.5,
     'score': 0.5,


In [89]:
improved_runs = simulate_grading(improved_rubric,students )

In [90]:
improved_runs

[[{'criteria_scores': [{'criterion': 'Politics - Category Alignment',
     'max_score': 0.25,
     'score': 0.25,
     'reason': 'The action (intelligence agents using SmartCity to gain information) is clearly motivated by political interests, as defined in the rubric.',
     'is_ambiguous': False},
    {'criterion': 'Politics - Action Specificity & Plausibility',
     'max_score': 0.25,
     'score': 0.25,
     'reason': 'The action is specific (intelligence agents using SmartCity installations in foreign countries for espionage) and plausible in the scenario.',
     'is_ambiguous': False},
    {'criterion': 'Politics - Scenario-Specific Negative Impact',
     'max_score': 0.25,
     'score': 0.25,
     'reason': 'The negative impact (loss of privacy and security for countries and their populations) is clearly described and specific to SmartCity stakeholders.',
     'is_ambiguous': False},
    {'criterion': 'Politics - Depth & Insight',
     'max_score': 0.25,
     'score': 0.1,
     

#### Compute metrics for both

In [91]:
baseline_eval = evaluation

In [92]:
baseline_eval

{'scores': {'ambiguity': 90.19,
  'discrimination': 18.35,
  'applicability': 44.44},
 'failures': ['Discrimination: Rubric does not separate strong and weak answers well.']}

In [93]:
improved_eval = detect_rubric_failures(improved_runs)

In [94]:
improved_eval

{'scores': {'ambiguity': 82.87,
  'discrimination': 23.15,
  'applicability': 55.56},
 'failures': ['Discrimination: Rubric does not separate strong and weak answers well.']}

In [96]:
def compare(baseline, improved):
    return {
        "ambiguity_improvement": improved["scores"]["ambiguity"] - baseline["scores"]["ambiguity"],
        "discrimination_improvement": improved["scores"]["discrimination"] - baseline["scores"]["discrimination"],
        "applicability_improvement": improved["scores"]["applicability"] - baseline["scores"]["applicability"]
    }

In [97]:
compare(baseline_eval, improved_eval)

{'ambiguity_improvement': -7.319999999999993,
 'discrimination_improvement': 4.799999999999997,
 'applicability_improvement': 11.120000000000005}